<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/RNNPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=6eeaf950651a9e30d9577cb07b228a6e798269e3ab93d485714c6f4ec3f1643d
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [8]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization,SimpleRNN,LSTM,Dense,Dropout,Input,Embedding,Bidirectional
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from lime.lime_text import LimeTextExplainer
import numpy as np

In [3]:
df = pd.read_csv('shglj.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'shglj.csv'

In [ ]:
train_df,test_df = train_test_split(df,test_size=0.3,random_state=42)

In [ ]:
# Separate the 'Fruit' from the 'Label'
train_texts = train_df['text'].values
train_labels = train_df['sentiment'].values

test_texts = test_df['text'].values
test_labels = test_df['sentiment'].values

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((train_texts, train_labels)).batch(32)
test_ds = tf.data.Dataset.from_tensor_slices((test_texts, test_labels)).batch(32)

In [ ]:
max_words = 10000
max_len = 250

vectorize_layer = TextVectorization(max_tokens=max_words,output_mode='int',output_sequence_length=max_len)

In [ ]:
vectorize_layer.adapt(train_texts) # adapt basically gives IDs to words

SimpleRNN

In [ ]:
model_rnn = Sequential(
    Input(shape=(1,),dtype=tf.string),
    vectorize_layer,

    Embedding(input_dim=max_words,output_dim=128), # embedding turn those ids to features

    SimpleRNN(64,return_sequences=True),
    SimpleRNN(64,return_sequences=False),

    Dense(128,activation='relu'),
    Dropout(0.2)
    Dense(1,activation='sigmoid')
)

LSTM

In [ ]:
model_lstm = Sequential(
    Input(shape=(1,),dtype=tf.string),
    vectorize_layer,

    LSTM(128,return_sequences=True),
    LSTM(68,return_sequences=False),

    Dense(128,activation='relu'),
    Dropout(0.2)
    Dense(1,activation='sigmoid')
)

Bidirectional: Here it read the text backward too, sometimes in the end of the sentences some words decide the sentiments.

In [ ]:
model_bidirection = Sequential(
    Input(shape=(1,),dtype=tf.string),
    vectorize_layer,

    Bidirectional(LSTM(68,return_sequences=False)),

    Dense(128,activation='relu'),
    Dropout(0.2)
    Dense(1,activation='sigmoid')
)

In [ ]:
model = model_rnn

In [ ]:
model.compile(optimization='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
model.fit(train_ds,validation_data=test_ds,epochs=10)

In [ ]:
def lime_predict(text):
  #first check if its string or not
  if isinstance(text,str):
    text = [text]

  input_array = np.array(text,dtype=object)

  preds = model.predict(input_array,verbose=0)

  return np.hstack([1 - preds, preds])

In [ ]:
explainer = LimeTextExplainer(class_names=['Negetive','Positive'])

ex = explainer.explain_instance(text_instance=test_text[0],lime_predict,num_features=10) # num_features means give me top 10 words from this sentence that make the prediction that particular output

# 4. THE SHOW: This creates the heatmap!
ex.show_in_notebook(text=True)